# Integración por Monte Carlo en FORTRAN
**Física Computacional — 106018C** · Lenguaje de cálculo: **FORTRAN 2008** · Visualización: Python

La semana pasada generamos números pseudoaleatorios y observamos sus patrones. Ahora usaremos muestras uniformes para calcular integrales. Empezaremos con ejemplos completos; al final usted completará tres programas: la convergencia del estimador de $\pi$, una integral por dos métodos y el efecto de un generador deficiente.

## Objetivos

- Transformar una muestra uniforme en $[0,1)$ a otro intervalo.
- Estimar un área contando aciertos y una integral promediando valores de una función.
- Reconocer por qué el error estadístico típico disminuye como $1/\sqrt{N}$.
- Programar y comentar los cálculos en FORTRAN; usar Python solo para las gráficas y la verificación.
- Relacionar la calidad de un generador con la calidad de una estimación física.

**Flujo:** ejecute los ejemplos en orden → complete los bloques `TODO` de los tres programas → compile y ejecute sus programas → interprete la gráfica → ejecute PASS/FAIL → redacte el informe de una página. **El notebook puede ejecutarse de principio a fin antes de completar los TODO; los resultados provisionales mostrarán FAIL hasta que los resuelva.**

## 0. Preparación de Google Colab

Colab ejecuta Python de forma nativa. Cada celda `%%writefile nombre.f90` guarda un programa FORTRAN; la siguiente lo compila con `gfortran` y ejecuta `./nombre`. Ejecute esta preparación una sola vez. Si `gfortran` ya está disponible, la instalación se omite.

In [ ]:
!which gfortran || (apt-get update -qq && apt-get install -y gfortran)
!gfortran --version | head -n 1

GNU Fortran (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;"><b>Flujo de trabajo:</b> Si cambia un programa FORTRAN, ejecute otra vez su celda <code>%%writefile</code> y después la celda de compilación. El signo <code>!</code> ejecuta un comando del sistema desde Colab. Los cálculos de las integrales se harán en FORTRAN.</div>

## 1. Generador de números aleatorios en Fortran: `random_number` y `random_seed`

Fortran trae un generador incorporado. La implementación depende del compilador; las versiones actuales de GNU Fortran documentan xoshiro256**:

| Instrucción | Qué hace |
|---|---|
| `call random_number(u)` | Asigna a `u` un real uniforme en $[0,1)$. |
| `call random_seed(size=n)` | Devuelve en `n` cuántos enteros forman el estado del generador. |
| `call random_seed(put=semilla)` | Fija el estado con el arreglo `semilla` de tamaño `n`. |

La inicialización por defecto depende de la implementación. Con una semilla fija, la secuencia se repite dentro de la misma implementación: esto permite depurar y verificar una simulación (Semana 4). No se garantiza que dos compiladores o versiones produzcan los mismos números.

El cambio de intervalo es el mismo de la Semana 4: si $u \in [0,1)$, entonces $x = A + (B-A)\,u \in [A, B)$.

In [ ]:
%%writefile semilla_demo.f90
! Semana 5 -- Semilla fija y cambio de intervalo con el generador de Fortran
program semilla_demo
  implicit none
  ! dp selecciona la precisión de los reales usados por el generador.
  integer, parameter :: dp = kind(1.0d0)
  integer :: i, n_sem
  integer, allocatable :: semilla(:)
  real(dp) :: u

  ! --- Tamaño del estado del generador y arreglo para la semilla ---
  call random_seed(size=n_sem)
  allocate(semilla(n_sem))
  print '(a, i0)', 'Enteros en el estado del generador: ', n_sem

  ! --- Primera secuencia con semilla 2026 ---
  ! Llenamos el estado de la semilla con un valor fijo para repetir el ejemplo.
  semilla = 2026
  call random_seed(put=semilla)
  print '(a)', 'Semilla 2026:'
  do i = 1, 3
     call random_number(u)
     print '(f12.8)', u
  end do

  ! --- Se fija otra vez la misma semilla: la secuencia se repite ---
  call random_seed(put=semilla)
  print '(a)', 'Semilla 2026 otra vez:'
  do i = 1, 3
     call random_number(u)
     print '(f12.8)', u
  end do

  ! La transformación lineal duplica el ancho del intervalo y desplaza su origen.
  ! --- Cambio de intervalo: [0,1) -> [-1,1) ---
  print '(a)', 'Uniformes en [-1,1):'
  do i = 1, 3
     call random_number(u)
     print '(f12.8)', 2.0_dp*u - 1.0_dp
  end do
end program semilla_demo

In [ ]:
!gfortran -O2 -fcheck=all semilla_demo.f90 -o semilla_demo && ./semilla_demo

## 2. Cálculo de $\pi$ por acierto y error

Se lanzan $N$ puntos uniformes en el cuadrado $[-1,1)\times[-1,1)$ (área 4). La fracción que cae dentro del círculo de radio 1 (área $\pi$) se aproxima a $p = \pi/4$, de modo que

$$\hat\pi = 4\,\frac{N_{\rm dentro}}{N}.$$

Un punto está dentro del círculo si $x^2 + y^2 < 1$; no hace falta calcular ninguna raíz. El programa cuenta los puntos, no los guarda, e imprime la estimación cada 1000 puntos.

In [ ]:
%%writefile pi_mc.f90
! Semana 5 -- Cálculo de pi por Monte Carlo (acierto y error)
! pi ~ 4 * aciertos / N, con N puntos uniformes en [-1,1) x [-1,1)
program pi_mc
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer :: n_total, n_aciertos, i, n_sem
  integer, allocatable :: semilla(:)
  real(dp) :: u, v, x, y, pi_est

  ! --- Semilla fija: la misma corrida se puede repetir ---
  call random_seed(size=n_sem)
  allocate(semilla(n_sem))
  semilla = 2026
  call random_seed(put=semilla)

  ! El círculo ocupa una fracción pi/4 del cuadrado de área 4.
  n_total    = 10000
  n_aciertos = 0                       ! acumulador (contador)

  ! --- Un punto por iteración; se cuenta, no se guarda ---
  do i = 1, n_total
     call random_number(u)             ! u en [0,1)
     call random_number(v)
     x = 2.0_dp*u - 1.0_dp             ! cambio de intervalo: [0,1) -> [-1,1)
     y = 2.0_dp*v - 1.0_dp
     if (x*x + y*y < 1.0_dp) n_aciertos = n_aciertos + 1   ! dentro del círculo

     ! --- Estimación parcial cada 1000 puntos ---
     ! mod(i,1000)=0 selecciona los múltiplos de 1000.
     if (mod(i, 1000) == 0) then
        ! real(...,dp) evita una división entera que truncaría la fracción.
        pi_est = 4.0_dp*real(n_aciertos, dp)/real(i, dp)
        print '(i8, f10.5)', i, pi_est
     end if
  end do
end program pi_mc

In [ ]:
!gfortran -O2 -fcheck=all pi_mc.f90 -o pi_mc && ./pi_mc

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;"><b>Compruebe la escala:</b> Si de 100 puntos, 79 caen dentro del círculo, la estimación es <code>4 × 79/100 = 3.16</code>. Al aumentar N, las estimaciones fluctúan; no tienen que acercarse a π de manera monótona.</div>

### Visualizar los puntos y la estimación parcial

La primera gráfica muestra puntos dentro y fuera del círculo; la segunda, cómo cambia la estimación a medida que se agregan puntos de cinco corridas independientes.

In [ ]:
%%writefile pi_trayectorias.f90
! Semana 5 -- Puntos de una corrida y evolución de pi_est para 5 semillas
! Salida: puntos.dat       (x, y, 1 si está dentro / 0 si está fuera)
!         trayectorias.dat (corrida, i, pi_est después de i puntos)
program pi_trayectorias
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: n_total = 20000, n_corridas = 5
  integer :: c, i, n_aciertos, n_sem, u_pts, u_tray, dentro
  integer, allocatable :: semilla(:)
  real(dp) :: u, v, x, y, pi_est

  ! La implementación decide cuántos enteros necesita para su semilla.
  call random_seed(size=n_sem)
  allocate(semilla(n_sem))
  ! newunit solicita unidades libres; replace evita mezclar corridas anteriores.
  open(newunit=u_pts,  file='puntos.dat',       status='replace')
  open(newunit=u_tray, file='trayectorias.dat', status='replace')

  ! --- Ciclo externo: corridas; ciclo interno: puntos ---
  do c = 1, n_corridas
     semilla = 77 + c                      ! una semilla distinta por corrida
     call random_seed(put=semilla)
     ! Cada trayectoria empieza con contador nuevo y una semilla distinta.
     n_aciertos = 0
     do i = 1, n_total
        call random_number(u);  call random_number(v)
        x = 2.0_dp*u - 1.0_dp;  y = 2.0_dp*v - 1.0_dp
        ! Indicador 0/1 de pertenencia al círculo; se suma al contador.
        dentro = 0
        if (x*x + y*y < 1.0_dp) dentro = 1
        n_aciertos = n_aciertos + dentro
        pi_est = 4.0_dp*real(n_aciertos, dp)/real(i, dp)
        ! Conservamos cada valor parcial para mostrar la convergencia.
        write(u_tray, '(i2, i7, f10.5)') c, i, pi_est
        ! --- Solo se guardan los puntos de la primera corrida ---
        if (c == 1 .and. i <= 3000) write(u_pts, '(2f10.5, i3)') x, y, dentro
     end do
  end do
  close(u_pts);  close(u_tray)
  print '(a)', 'Archivos escritos: puntos.dat y trayectorias.dat'
end program pi_trayectorias

In [ ]:
!rm -f puntos.dat trayectorias.dat
!gfortran -O2 -fcheck=all pi_trayectorias.f90 -o pi_trayectorias && ./pi_trayectorias

In [ ]:
# Gráfica de los puntos y de las trayectorias de pi_est (lámina "Muestreo y convergencia")
import numpy as np
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, GRIS = "#162A54", "#C9972C", "#B02A20", "#8A8A8A"

pts = np.loadtxt("puntos.dat")
tray = np.loadtxt("trayectorias.dat")
dentro = pts[:, 2] == 1

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [1, 1.5]})

# --- Panel izquierdo: puntos dentro y fuera del círculo ---
ax[0].scatter(pts[dentro, 0], pts[dentro, 1], s=3, color=AZUL, label="dentro")
ax[0].scatter(pts[~dentro, 0], pts[~dentro, 1], s=3, color=DORADO, label="fuera")
t = np.linspace(0, 2*np.pi, 300)
ax[0].plot(np.cos(t), np.sin(t), "k", lw=1)
ax[0].set_aspect("equal")
ax[0].set_title(f"N = {len(pts)} puntos: π ≈ {4*dentro.mean():.3f}")

# --- Panel derecho: pi_est contra N para las 5 semillas ---
for c in range(1, 6):
    s = tray[tray[:, 0] == c]
    ax[1].plot(s[:, 1], s[:, 2], lw=1, color=AZUL if c == 1 else GRIS)
ax[1].axhline(np.pi, color=ROJO, ls="--", lw=1.2)
ax[1].set_xscale("log"); ax[1].set_xlim(10, 2e4); ax[1].set_ylim(2.8, 3.5)
ax[1].set_xlabel("número de puntos $N$"); ax[1].set_ylabel("estimación de π")
ax[1].set_title("Cinco corridas con semillas distintas")
plt.tight_layout(); plt.show()

## 3. Error estadístico del estimador

Cada lanzamiento define una variable de Bernoulli: $O_i = 1$ si el punto cae dentro (probabilidad $p = \pi/4$) y $O_i = 0$ si cae fuera. El estimador es un promedio,

$$\hat\pi = 4\,\langle O\rangle = 4\,\frac{1}{N}\sum_{i=1}^{N} O_i,$$

con la misma estructura que $\langle x\rangle$ en las pruebas del generador de la Semana 4. Para una variable de Bernoulli, $\sigma_O^2 = \langle O^2\rangle - \langle O\rangle^2 = p - p^2 = p(1-p)$. El promedio de $N$ valores independientes tiene desviación $\sigma_O/\sqrt{N}$, así que

$$\sigma_{\hat\pi} = \frac{4\sqrt{p(1-p)}}{\sqrt{N}} = \frac{1.642}{\sqrt{N}}.$$

La celda siguiente calcula esta predicción y la compara con la dispersión de las 5 trayectorias de la sección 2 en algunos valores de $N$.

**Dos tamaños distintos:** N es el número de puntos dentro de cada estimación; M es el número de corridas. La desviación entre corridas estima la incertidumbre de **una** estimación. La incertidumbre de la media de M corridas es aproximadamente esa desviación dividida por √M. Las trayectorias acumuladas contienen puntos correlacionados entre sí; para cada N se compara una estimación de cada corrida.

El TCL explica por qué la distribución de los promedios se aproxima a una gaussiana para N grande, suponiendo independencia y varianza finita. La relación σ/√N para la desviación de la media no requiere que los datos originales sean gaussianos. El Anexo A permite explorar esa distribución gráficamente.

In [ ]:
# Predicción 1.642/sqrt(N) contra la dispersión de las 5 corridas de la sección 2
import numpy as np

p = np.pi/4
coef = 4*np.sqrt(p*(1 - p))
print(f"4*sqrt(p(1-p)) = {coef:.4f}\n")

tray = np.loadtxt("trayectorias.dat")
print("     N   sigma predicha   desviación de las 5 corridas")
for N in [100, 1000, 10000, 20000]:
    valores = tray[tray[:, 1] == N][:, 2]          # pi_est de cada corrida en ese N
    print(f"{N:6d}   {coef/np.sqrt(N):12.4f}   {valores.std(ddof=1):14.4f}")

<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;"><b>Distinga dos tamaños:</b> <code>N</code> es el número de puntos de una estimación; <code>M</code> es el número de corridas. La dispersión entre corridas describe una estimación típica. La incertidumbre de su media es esa dispersión dividida por la raíz de M.</div>

**Consecuencia práctica.** Como $\sigma \propto 1/\sqrt{N}$, una cifra decimal más (reducir $\sigma$ por 10) exige **100 veces más puntos**. Para una precisión $\varepsilon$ se necesitan $N \approx (1.642/\varepsilon)^2$ puntos: $\sim 3\times10^4$ para $\varepsilon = 10^{-2}$ y $\sim 3\times10^8$ para $\varepsilon = 10^{-4}$.

## 4. Integración por Monte Carlo: método de valor medio

Si $x$ es uniforme en $[a,b)$, su densidad es $P(x) = 1/(b-a)$ y

$$I = \int_a^b f(x)\,dx = (b-a)\int_a^b P(x)\,f(x)\,dx = (b-a)\,\langle f\rangle \approx \frac{b-a}{N}\sum_{k=1}^{N} f(x_k).$$

El error es $\sigma_I = (b-a)\,\sigma_f/\sqrt{N}$, con $\sigma_f^2 = \langle f^2\rangle - \langle f\rangle^2$.

**Ejemplo.** $I = \int_0^1 x^2\,dx = 1/3$. Aquí $\langle f\rangle = 1/3$ y $\langle f^2\rangle = \int_0^1 x^4\,dx = 1/5$, de modo que $\sigma_f = \sqrt{1/5 - 1/9} = 0.298$ y el error esperado es $0.298/\sqrt{N}$.

El método de acierto y error de la sección 2 también calcula integrales: se lanzan puntos en un rectángulo $[a,b)\times[0,H)$ que contiene la curva, suponiendo $0\le f(x)\le H$ en todo el intervalo y se cuentan los que quedan debajo, $I \approx (b-a)\,H\,N_{\rm bajo}/N$.

In [ ]:
%%writefile integral_x2.f90
! Semana 5 -- Integral de x^2 en [0,1) por el método de valor medio (valor exacto: 1/3)
program integral_x2
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  ! sigma_f es la desviación de x^2 en [0,1]; sigma de la integral ~ sigma_f/sqrt(N).
  real(dp), parameter :: a = 0.0_dp, b = 1.0_dp, sigma_f = 0.298142_dp
  integer :: k, n_sem
  integer(8) :: i, n_total
  integer, allocatable :: semilla(:)
  real(dp) :: u, x, suma_f, integral

  ! Una semilla fija facilita reproducir la demostración con el mismo compilador.
  call random_seed(size=n_sem)
  allocate(semilla(n_sem))
  semilla = 2026
  call random_seed(put=semilla)

  print '(a)', '        N    integral       |error|   0.298/sqrt(N)   |error|/sigma'
  ! Repetimos para N = 10^2, 10^3, ..., 10^6; el generador sigue avanzando.
  do k = 2, 6
     n_total = 10_8**k
     suma_f = 0.0_dp                       ! acumulador de f(x_k)
     do i = 1, n_total
        call random_number(u)
        x = a + (b - a)*u                  ! x uniforme en [a,b)
        suma_f = suma_f + x*x              ! f(x) = x^2
     end do
     ! La integral es ancho del intervalo por promedio de f(x).
     integral = (b - a)*suma_f/real(n_total, dp)
     ! Comparamos error observado con la escala estadística predicha,
     ! sin esperar que el cociente error/sigma sea exactamente uno.
     print '(i9, f12.6, 2f14.6, f12.2)', n_total, integral, abs(integral - 1.0_dp/3.0_dp), &
           sigma_f/sqrt(real(n_total, dp)), &
           abs(integral - 1.0_dp/3.0_dp)/(sigma_f/sqrt(real(n_total, dp)))
  end do
end program integral_x2

In [ ]:
!gfortran -O2 -fcheck=all integral_x2.f90 -o integral_x2 && ./integral_x2

<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;"><b>Evite la división entera:</b> Al formar un promedio o una fracción en FORTRAN convierta el contador con <code>real(n,dp)</code> o utilice un numerador real. Una división de dos enteros descarta la parte fraccionaria.</div>

## 5. Efecto del generador: el LCG $(57, 1, 256)$

En la Semana 4 se vio que el generador lineal congruente

$$r_{i+1} = (a\,r_i + c) \bmod M, \qquad x_i = r_i/M, \qquad (a, c, M) = (57, 1, 256)$$

tiene período máximo (256), pero sus pares consecutivos $(x_i, x_{i+1})$ caen en unas pocas líneas paralelas. En Fortran, el LCG se programa con aritmética entera y la función `mod`. El programa siguiente escribe en `lcg_pares.dat` los puntos $(x, y)$ en $[-1,1)^2$ que se forman tomando los números de dos en dos, como en el programa de $\pi$.

In [ ]:
%%writefile lcg_pares.f90
! Semana 5 -- Puntos (x, y) formados con números consecutivos del LCG (57, 1, 256)
program lcg_pares
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  ! El estado toma solo 256 valores enteros; un módulo tan pequeño deja patrones.
  integer, parameter :: a_lcg = 57, c_lcg = 1, M_lcg = 256
  integer :: i, r, u_dat
  real(dp) :: x, y

  r = 10                                         ! semilla del LCG
  ! Escribimos las parejas para graficar su distribución en el plano.
  open(newunit=u_dat, file='lcg_pares.dat', status='replace')
  do i = 1, 1000
     r = mod(a_lcg*r + c_lcg, M_lcg)             ! primer número del par
     ! Normalizamos r/M a [0,1) y luego lo llevamos a [-1,1).
     x = 2.0_dp*real(r, dp)/M_lcg - 1.0_dp
     ! Avanzar el estado otra vez da la segunda coordenada del mismo punto.
     r = mod(a_lcg*r + c_lcg, M_lcg)             ! segundo número del par
     y = 2.0_dp*real(r, dp)/M_lcg - 1.0_dp
     write(u_dat, '(2f10.5)') x, y
  end do
  close(u_dat)
  print '(a)', 'Archivo escrito: lcg_pares.dat (1000 puntos)'
end program lcg_pares

In [ ]:
!rm -f lcg_pares.dat
!gfortran -O2 -fcheck=all lcg_pares.f90 -o lcg_pares && ./lcg_pares

In [ ]:
# Puntos del LCG (57, 1, 256) frente a puntos de random_number
import numpy as np
import matplotlib.pyplot as plt

AZUL, ROJO = "#162A54", "#B02A20"
lcg = np.loadtxt("lcg_pares.dat")
pts = np.loadtxt("puntos.dat")[:1000]
n_distintos = len(np.unique(lcg, axis=0))

fig, ax = plt.subplots(1, 2, figsize=(9, 4.4))
t = np.linspace(0, 2*np.pi, 300)
for a_, datos, color, titulo in [(ax[0], pts, AZUL, "random_number: 1000 puntos"),
                                 (ax[1], lcg, ROJO, f"LCG (57,1,256): {n_distintos} puntos distintos")]:
    a_.scatter(datos[:, 0], datos[:, 1], s=6, color=color)
    a_.plot(np.cos(t), np.sin(t), "k", lw=1)
    a_.set_aspect("equal"); a_.set_title(titulo)
plt.tight_layout(); plt.show()

El LCG del ejemplo tiene pocos estados: aunque produce valores en el intervalo correcto, sus parejas repiten un patrón. La observación conecta el generador de la semana anterior con el muestreo que acabamos de emplear. En la Parte C del taller se calcula $\hat\pi$ con estos puntos.

# Taller 5 — Cálculo de $\pi$ y convergencia

## Propósito

Mida en FORTRAN cómo disminuye el error del estimador de $\pi$ al aumentar $N$, compare dos métodos para calcular $\int_0^{\pi/2}\sin x\,dx = 1$ y analice el efecto de un generador deficiente sobre la estimación de $\pi$. No se requieren archivos de entrada externos.

Las plantillas fijan parámetros, declaran variables y escriben los resultados. Los bloques `TODO` describen lo que debe obtener, remitiendo a las secciones anteriores; la forma de programarlo la decide usted. Mientras estén vacíos, las salidas usan el valor provisional **−1** para indicar que la estimación aún no se ha calculado. Así puede ejecutar las celdas en orden sin producir valores indefinidos. PASS/FAIL debe mostrar FAIL hasta completar el trabajo.

## Parte A — Convergencia del estimador de $\pi$

Para cada $N = 10^2, 10^3, \ldots, 10^7$, repita el cálculo de $\hat\pi$ de la sección 2 con $M = 20$ semillas distintas. Si las estimaciones son $\hat\pi_1, \ldots, \hat\pi_M$, su media y su desviación estándar muestral son
$$\bar\pi = \frac1M\sum_{j=1}^M \hat\pi_j, \qquad
s = \sqrt{\frac{M}{M-1}\left(\frac1M\sum_{j=1}^M \hat\pi_j^2 - \bar\pi^2\right)}.$$
Distinga $s$, que describe una corrida típica, de $s/\sqrt M$, que describe la incertidumbre de la media de las $M$ corridas. La sección 3 predice $s \approx 1.642/\sqrt{N}$.

Si el error decrece como $1/\sqrt{N}$, los puntos $(\log_{10} N, \log_{10} s)$ caen sobre una recta de pendiente $-1/2$. La pendiente de $n$ puntos $(X_k, Y_k)$ por mínimos cuadrados (Semana 3) es
$$m = \frac{n\sum X_k Y_k - \sum X_k \sum Y_k}{n\sum X_k^2 - \left(\sum X_k\right)^2}.$$

Complete los **cinco bloques TODO** de `taller5_pi.f90`. El programa guarda en `convergencia.dat` el valor de $N$, la media y la desviación, y en `pendiente.dat` la pendiente $m$. El caso $N = 10^7$ tarda unos segundos. Al depurar puede reducir temporalmente el límite superior del ciclo en `k`; restáurelo a 7 antes de entregar.

In [ ]:
%%writefile taller5_pi.f90
! Taller 5 -- Convergencia del estimador de pi por acierto y error.
! Salidas: convergencia.dat (N, media, desviación) y pendiente.dat (pendiente m).
program taller5_pi
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: M = 20                  ! corridas por cada N
  integer :: k, j, n_sem, u_dat, n_puntos
  integer(8) :: i, n_total, n_aciertos
  integer, allocatable :: semilla(:)
  real(dp) :: u, v, x, y, pi_est, suma, suma2, media, desv
  real(dp) :: X_k, Y_k, sx, sy, sxx, sxy, pendiente

  call random_seed(size=n_sem)
  allocate(semilla(n_sem))

  sx = 0.0_dp; sy = 0.0_dp; sxx = 0.0_dp; sxy = 0.0_dp; n_puntos = 0   ! sumas del ajuste
  open(newunit=u_dat,file='convergencia.dat',status='replace')
  do k = 2, 7                                   ! N = 10^2, ..., 10^7
     n_total = 10_8**k
     suma = 0.0_dp; suma2 = 0.0_dp              ! sumas sobre las M corridas
     do j = 1, M
        semilla = 1000*k + j                    ! una semilla distinta por corrida
        call random_seed(put=semilla)
        pi_est = -1.0_dp                        ! valor provisional

        ! TODO A1: obtenga pi_est con n_total puntos uniformes en el cuadrado (sección 2).

        ! TODO A2: acumule lo necesario para la media y la desviación de las M corridas.
     end do

     media = -1.0_dp; desv = -1.0_dp            ! valores provisionales
     ! TODO A3: calcule la media y la desviación estándar muestral de las M estimaciones.

     write(u_dat,'(i10,2es25.16e3)') n_total,media,desv
     print '(i10,2f12.6)', n_total,media,desv

     ! TODO A4: acumule lo necesario para ajustar log10(desv) contra log10(N).
  end do
  close(u_dat)

  pendiente = -1.0_dp                           ! valor provisional
  ! TODO A5: calcule la pendiente del ajuste por mínimos cuadrados.

  open(newunit=u_dat,file='pendiente.dat',status='replace')
  write(u_dat,'(es25.16e3)') pendiente
  close(u_dat)
  print '(a,f9.4)', 'Pendiente de log10(s) contra log10(N): ', pendiente
end program taller5_pi

In [ ]:
!rm -f convergencia.dat pendiente.dat
!gfortran -std=f2008 -O2 -fcheck=all -o taller5_pi taller5_pi.f90 && ./taller5_pi

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;"><b>Prueba manual:</b> con $N = 100$ cada estimación de $\pi$ debe estar entre 0 y 4, y la desviación de las 20 corridas debería ser del orden de $0.16$. Con $N = 10^7$, la media debería coincidir con $\pi$ en al menos tres decimales.</div>

### Visualización de la Parte A

La gráfica usa exclusivamente `convergencia.dat` producido por FORTRAN. Compare la desviación medida con la predicción $1.642/\sqrt{N}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Lee N, media y desviación calculadas por el programa FORTRAN.
N, media, desv = np.loadtxt('convergencia.dat', ndmin=2).T

plt.figure(figsize=(7,4.5))
plt.loglog(N, 1.642/np.sqrt(N), '--', label=r'predicción $1.642/\sqrt{N}$')
if np.any(desv > 0):
    plt.loglog(N[desv > 0], desv[desv > 0], 'o', label='desviación de 20 corridas (FORTRAN)')
plt.xlabel('Número de puntos N')
plt.ylabel('Desviación estándar de la estimación de π')
plt.grid(True, which='both', alpha=.25)
plt.legend()
plt.tight_layout()
plt.savefig('convergencia_pi.png', dpi=180)
plt.show()

## Parte B — Integral en 1D por dos métodos

En el método de **valor medio**, tome $x=a+(b-a)u$ con $u$ uniforme en $[0,1)$ y estime
$$I_\mathrm{VM}=\frac{b-a}{N}\sum_{i=1}^{N}\sin x_i.$$

En **acierto y error**, construya puntos $(x,y)$ uniformes en el rectángulo $[0,\pi/2)\times[0,1)$, cuente los puntos que cumplen $y<\sin x$ y estime
$$I_\mathrm{AE}=(b-a)\frac{N_\mathrm{bajo}}{N}.$$

Para comparar los métodos, repita cada cálculo $R$ veces y obtenga la media y la desviación estándar muestral de las $R$ estimaciones, con las mismas fórmulas de la Parte A.

Complete los **cuatro bloques TODO** de `taller5_seno.f90`. Cada repetición calcula las dos integrales con $N=1000$ puntos; se realizan $R=1000$ repeticiones para medir la dispersión de los estimadores. El programa guarda en `seno.dat` la media y la desviación estándar muestral de cada método. Compare las medias con 1 y las dispersiones con $0.483/\sqrt{N}$ (valor medio) y $0.756/\sqrt{N}$ (acierto y error). Al depurar puede disminuir temporalmente `R`; restáurelo a 1000 antes de entregar.

In [ ]:
%%writefile taller5_seno.f90
! Taller 5 -- Integral de sin(x) en [0,pi/2) por valor medio y acierto y error.
! Salida: seno.dat (media_vm, desv_vm, media_ae, desv_ae).
program taller5_seno
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: N = 1000, R = 1000      ! muestras por corrida y repeticiones
  real(dp), parameter :: a = 0.0_dp, b = 2.0_dp*atan(1.0_dp), H = 1.0_dp
  integer :: rep, i, n_bajo, n_sem, u_dat
  integer, allocatable :: semilla(:)
  real(dp) :: u, v, x, y, suma_f, I_vm, I_ae
  real(dp) :: s_vm, s2_vm, s_ae, s2_ae, media_vm, desv_vm, media_ae, desv_ae

  ! El tamaño del arreglo de semilla depende de la implementación.
  call random_seed(size=n_sem)
  allocate(semilla(n_sem))
  semilla = 314
  call random_seed(put=semilla)                ! se fija una vez, antes de las R corridas

  s_vm = 0.0_dp; s2_vm = 0.0_dp; s_ae = 0.0_dp; s2_ae = 0.0_dp
  do rep = 1, R
     ! Valores provisionales: reemplácelos dentro de los bloques TODO.
     I_vm = -1.0_dp
     I_ae = -1.0_dp

     ! TODO B1: obtenga I_vm por valor medio usando N puntos (sección 4).

     ! TODO B2: obtenga I_ae por acierto y error en [a,b) x [0,H) (sección 4).

     ! TODO B3: acumule las dos integrales y sus cuadrados para las R corridas.
  end do

  ! Los valores -1 indican que aún no se calcularon las estadísticas.
  media_vm = -1.0_dp; desv_vm = -1.0_dp
  media_ae = -1.0_dp; desv_ae = -1.0_dp
  ! TODO B4: calcule media y desviación estándar muestral para cada método.

  ! Cuatro números sin encabezado para la lectura automática.
  open(newunit=u_dat,file='seno.dat',status='replace')
  write(u_dat,'(4es25.16e3)') media_vm,desv_vm,media_ae,desv_ae
  close(u_dat)
  print '(a,2f10.5)', 'Valor medio (media, desviacion): ',media_vm,desv_vm
  print '(a,2f10.5)', 'Acierto y error (media, desviacion): ',media_ae,desv_ae
end program taller5_seno

In [ ]:
!rm -f seno.dat
!gfortran -std=f2008 -O2 -fcheck=all -o taller5_seno taller5_seno.f90 && ./taller5_seno

<div style="background-color:#eef7ee; border-left:5px solid #4caf50; padding:10px 15px; margin:10px 0;"><b>Prueba manual:</b> En una repetición completa, cada integral debe estar entre 0 y π/2. Al terminar, las dos medias deberían ser próximas a 1. Las desviaciones entre corridas deben ser positivas y no serán iguales para los dos métodos.</div>

## Parte C — Efecto del generador deficiente

Repita el cálculo de $\hat\pi$ de la sección 2 para $N = 10^2, \ldots, 10^7$, con una sola corrida por cada $N$, pero tome las coordenadas $(x, y)$ de números consecutivos del LCG $(57, 1, 256)$ de la sección 5, con semilla $r = 10$, en lugar de `random_number`.

Complete el **bloque TODO** de `taller5_lcg.f90`. El programa guarda en `lcg.dat` el valor de $N$ y la estimación $\hat\pi$. Compare cómo cambia el error con $N$ frente a lo observado en la Parte A.

In [ ]:
%%writefile taller5_lcg.f90
! Taller 5 -- pi por acierto y error con el LCG (57, 1, 256) en lugar de random_number.
! Salida: lcg.dat (N, pi_est).
program taller5_lcg
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: a_lcg = 57, c_lcg = 1, M_lcg = 256
  integer :: k, r, u_dat
  integer(8) :: i, n_total, n_aciertos
  real(dp) :: x, y, pi_est

  open(newunit=u_dat,file='lcg.dat',status='replace')
  do k = 2, 7                                   ! N = 10^2, ..., 10^7
     n_total = 10_8**k
     r = 10                                     ! semilla del LCG, la misma para cada N
     pi_est = -1.0_dp                           ! valor provisional

     ! TODO C1: obtenga pi_est con n_total puntos cuyas coordenadas salen del LCG (sección 5).

     write(u_dat,'(i10,es25.16e3)') n_total,pi_est
     print '(i10,f12.6)', n_total,pi_est
  end do
  close(u_dat)
end program taller5_lcg

In [ ]:
!rm -f lcg.dat
!gfortran -std=f2008 -O2 -fcheck=all -o taller5_lcg taller5_lcg.f90 && ./taller5_lcg

<div style="background-color:#fdecea; border-left:5px solid #e53935; padding:10px 15px; margin:10px 0;"><b>Interprete el error con cuidado:</b> La desviación $s$ estima la escala típica de fluctuación, no un límite que cada corrida deba respetar. Por eso la Parte A mide $s$ con 20 corridas en lugar de usar una sola. En la Parte C, en cambio, el error no es una fluctuación: es sistemático.</div>

## Parte D — Interpretación

Responda brevemente en el notebook:

1. Con la pendiente de la Parte A, ¿cuántos puntos se necesitarían para obtener $\pi$ con un error de $10^{-6}$? ¿Es práctico?
2. Con el mismo $N$, ¿cuál método de la Parte B tiene menor dispersión? ¿Por qué? ¿Bastaría una sola corrida para afirmarlo?
3. ¿Por qué, con el LCG de la Parte C, aumentar $N$ no reduce el error, aunque cada número esté en $[0,1)$?
4. ¿Por qué el cálculo de $\pi$ es una integral en 2D, aunque su estimador se escriba como una fracción?

## Verificación final PASS/FAIL

Ejecute esta celda **después** de volver a guardar, compilar y correr los tres programas del taller. Comprueba que los archivos fueron producidos por los `.f90` actuales y compara los resultados con valores de referencia, permitiendo fluctuaciones estadísticas. Antes de resolver los TODO la celda debe mostrar FAIL, sin detener el notebook. Un PASS no sustituye la revisión de su código ni de sus explicaciones.

In [ ]:
from pathlib import Path
import numpy as np

veredictos = []
def informar(nombre, condicion, detalle=''):
    estado = 'PASS' if condicion else 'FAIL'
    veredictos.append(bool(condicion))
    print(f'{estado} | {nombre}' + (f' — {detalle}' if detalle else ''))

def leer_salida(archivo, fuente, columnas):
    ruta, programa = Path(archivo), Path(fuente)
    if not programa.exists() or not ruta.exists():
        raise ValueError(f'Falta {archivo} o {fuente}; ejecute ambas celdas del programa.')
    if ruta.stat().st_mtime < programa.stat().st_mtime:
        raise ValueError(f'{archivo} es anterior a {fuente}; compile y ejecute otra vez.')
    datos = np.loadtxt(ruta, ndmin=2)
    if datos.shape[1] != columnas or not np.isfinite(datos).all():
        raise ValueError(f'Formato o valores incorrectos en {archivo}.')
    return datos

niveles = 10.0**np.arange(2, 8)                # N = 10^2, ..., 10^7

# --- Parte A: media y dispersión de 20 corridas; pendiente calculada en FORTRAN ---
try:
    filas = leer_salida('convergencia.dat','taller5_pi.f90',3)
    if filas.shape != (6,3) or not np.allclose(filas[:,0], niveles):
        raise ValueError('Se requieren seis filas con N=10^2,...,10^7.')
    n, media, desv = filas.T
    sigma = 1.642/np.sqrt(n)
    ok = (abs(media[4]-np.pi) < 6*sigma[4]/np.sqrt(20)
          and np.all((.45*sigma < desv) & (desv < 1.6*sigma)))
    informar('Convergencia de pi', ok,
             f'media(N=10^6)={media[4]:.6f}, s(N=10^6)={desv[4]:.6f}, predicción={sigma[4]:.6f}')
except Exception as error:
    informar('Convergencia de pi',False,str(error))

try:
    n, media, desv = leer_salida('convergencia.dat','taller5_pi.f90',3).T
    m = leer_salida('pendiente.dat','taller5_pi.f90',1)[0,0]
    if len(n) != 6 or not np.all(desv > 0):
        raise ValueError('convergencia.dat debe tener seis desviaciones positivas.')
    m_datos = np.polyfit(np.log10(n), np.log10(desv), 1)[0]
    informar('Pendiente del ajuste en FORTRAN', -.6 < m < -.4 and abs(m-m_datos) < 1e-4,
             f'm={m:.4f}, predicción=-0.5')
except Exception as error:
    informar('Pendiente del ajuste en FORTRAN',False,str(error))

# --- Parte B: integral de sin x por dos métodos ---
try:
    filas = leer_salida('seno.dat','taller5_seno.f90',4)
    if filas.shape != (1,4):
        raise ValueError('seno.dat debe contener media y desviación de cada método.')
    vm, svm, ae, sae = filas[0]
    n, repeticiones = 1000,1000
    sigma_vm = np.sqrt(np.pi**2/8-1)/np.sqrt(n)
    sigma_ae = np.sqrt(np.pi/2-1)/np.sqrt(n)
    ok_vm = (abs(vm-1)<6*sigma_vm/np.sqrt(repeticiones)
             and .65*sigma_vm < svm < 1.35*sigma_vm)
    ok_ae = (abs(ae-1)<6*sigma_ae/np.sqrt(repeticiones)
             and .65*sigma_ae < sae < 1.35*sigma_ae)
    informar('Integral 1D, valor medio',ok_vm,
             f'media={vm:.6f}, dispersión={svm:.6f}')
    informar('Integral 1D, acierto y error',ok_ae,
             f'media={ae:.6f}, dispersión={sae:.6f}')
except Exception as error:
    informar('Integral 1D, valor medio',False,str(error))
    informar('Integral 1D, acierto y error',False,'Revise el mismo programa.')

# --- Parte C: con el LCG la estimación se estanca lejos de pi ---
try:
    filas = leer_salida('lcg.dat','taller5_lcg.f90',2)
    if filas.shape != (6,2) or not np.allclose(filas[:,0], niveles):
        raise ValueError('Se requieren seis filas con N=10^2,...,10^7.')
    pi_lcg = filas[:,1]
    error_lcg = abs(pi_lcg - np.pi)
    ok = .02 < error_lcg[-1] < .2 and abs(pi_lcg[-1]-pi_lcg[-3]) < 1e-3
    informar('Generador deficiente', ok,
             f'|error| con N=10^5: {error_lcg[-3]:.4f}; con N=10^7: {error_lcg[-1]:.4f}')
except Exception as error:
    informar('Generador deficiente',False,str(error))

print(f'\nRESUMEN: {sum(veredictos)}/{len(veredictos)} PASS')
print('Las explicaciones científicas, los comentarios del código y el informe se revisan aparte.')

## Informe de una página y entrega

Use la plantilla LaTeX del curso. Organice el informe en **contexto, método, resultados, obstáculo o decisión de implementación, conclusión y uso de IA**. En resultados incluya la figura de convergencia de la Parte A con la pendiente obtenida, las medias y desviaciones de los dos métodos de la Parte B, el resultado con el LCG de la Parte C y el resumen PASS/FAIL. Explique qué aprende al comparar métodos mediante varias corridas.

**Entregue:** notebook completado y ejecutado, `taller5_pi.f90`, `taller5_seno.f90`, `taller5_lcg.f90`, `convergencia.dat`, `pendiente.dat`, `seno.dat`, `lcg.dat`, `convergencia_pi.png` e informe PDF de una página. No necesita adjuntar los programas compilados ni los archivos de los ejemplos guiados.

### Lista de verificación

- [ ] Los ejemplos completos se ejecutan desde una sesión nueva de Colab.
- [ ] Completé los diez bloques `TODO` y puedo explicar cada uno.
- [ ] Los cálculos de $\pi$, de las integrales y de las desviaciones se hacen en FORTRAN.
- [ ] Volví a ejecutar `%%writefile` y la compilación después de modificar cada programa.
- [ ] Los cinco controles numéricos muestran PASS y respondí las preguntas de interpretación.
- [ ] Mi informe ocupa una página y cita los resultados de mi propia corrida.

## Anexo A — Teorema central del límite (opcional)

El promedio de $N$ números independientes, con media $\mu$ y varianza $\sigma^2$, tiende a una gaussiana con desviación $\sigma/\sqrt{N}$, sin importar la distribución original. Para la uniforme en $[0,1)$: $\mu = 1/2$ y $\sigma^2 = 1/12$. El programa guarda 100 000 promedios de $N = 1, 2, 5$ y $20$ números de `random_number`.

In [ ]:
%%writefile tcl_uniforme.f90
! Anexo A -- Promedios de N uniformes (teorema central del límite)
program tcl_uniforme
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: R = 100000              ! número de promedios por cada N
  integer :: rep, c, j, n_sem, u_dat
  integer :: lista_N(4) = [1, 2, 5, 20]
  integer, allocatable :: semilla(:)
  real(dp) :: u, suma, promedio(4)

  call random_seed(size=n_sem)
  allocate(semilla(n_sem))
  semilla = 2026
  call random_seed(put=semilla)

  open(newunit=u_dat, file='tcl.dat', status='replace')
  do rep = 1, R
     do c = 1, 4                                ! un promedio para cada N
        suma = 0.0_dp
        do j = 1, lista_N(c)
           call random_number(u);  suma = suma + u
        end do
        promedio(c) = suma/lista_N(c)
     end do
     write(u_dat, '(4f10.6)') promedio
  end do
  close(u_dat)
end program tcl_uniforme

In [ ]:
!gfortran -O2 -fcheck=all tcl_uniforme.f90 -o tcl_uniforme && ./tcl_uniforme

In [ ]:
# Histogramas de los promedios y gaussiana del teorema
import numpy as np
import matplotlib.pyplot as plt
datos = np.loadtxt("tcl.dat")
xs = np.linspace(0, 1, 400)
plt.figure(figsize=(7.5, 4.4))
for col, (N, color) in enumerate(zip([1, 2, 5, 20], ["#8A8A8A", "#C9972C", "#B02A20", "#162A54"])):
    plt.hist(datos[:, col], bins=np.linspace(0, 1, 81), density=True, histtype="step",
             lw=1.6, color=color, label=f"N = {N}")
    if N >= 5:
        s = np.sqrt(1/12)/np.sqrt(N)
        plt.plot(xs, np.exp(-(xs - 0.5)**2/(2*s**2))/(np.sqrt(2*np.pi)*s), "k--", lw=1)
plt.xlabel(r"$\bar x$"); plt.yticks([]); plt.legend(frameon=False)
plt.title("Promedios de N números uniformes (línea punteada: gaussiana)")
plt.tight_layout(); plt.show()